# 第 7 章 · 软件框架：ROS 2、中间件与实时系统

配套讲义：仓库里 `robot-system-architecture/notes/lessons/07-ros2-middleware.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

（QoS 的数字照动画：每时刻发 1 条、链路 1.25 次/时刻；坐标变换的位姿是编的）

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 公式表 重发次数 | $1/(1-p)$ | `expected_sends`、`monte_carlo_sends` | p = 0.35：公式 1.538，模拟 1.536 |
| 公式表 队列不爆 | 速率 × $1/(1-p)$ < 容量 | `queue_ok(1.0, p, 1.25)` | p = 0.05 能跟上；0.35 跟不上；临界 0.20 |
| 第 3 步 / 动画 | RELIABLE vs BEST_EFFORT | `simulate_qos(p)` | p = 0.35：RELIABLE 旧 31 个时刻，BEST_EFFORT 旧 1 个、收到 60% |
| 第 4 步 tf 变换链 | ${}^AT_C = {}^AT_B\,{}^BT_C$ | `se2(...) @ p` | 相机系 (1, 0) → base_link 系 (0.2, 1.0)，逆变换变回去 |

QoS 的数字照动画来：每个时刻发 1 条、链路每个时刻最多发 1.25 次、丢包率 35%。坐标变换的位姿是编的。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `expected_sends`**：可靠传输每条平均要发 1 / (1 − p) 次。

In [ ]:
def expected_sends(p: float) -> float:
    """可靠传输每条平均要发 1 / (1 − p) 次。"""
    return 1.0 / (1.0 - p)

**函数 `monte_carlo_sends`**：模拟：每条消息一直重发直到送达，数一数平均发了几次。

In [ ]:
def monte_carlo_sends(p: float, n: int = 200_000, seed: int = 0) -> float:
    """模拟：每条消息一直重发直到送达，数一数平均发了几次。"""
    rng = np.random.default_rng(seed)
    return float(rng.geometric(1.0 - p, size=n).mean())

**函数 `queue_ok`**：队列不爆的条件：发布速率 × 1/(1−p) < 链路容量。

In [ ]:
def queue_ok(rate: float, p: float, capacity: float) -> bool:
    """队列不爆的条件：发布速率 × 1/(1−p) < 链路容量。"""
    return rate * expected_sends(p) < capacity

**函数 `simulate_qos`**：每个时刻发布 1 条。RELIABLE：排队、丢了重发；BEST_EFFORT：只发最新的，丢了就丢。返回最后的"数据年龄"（时刻数）。

In [ ]:
def simulate_qos(p: float, ticks: int = 200, capacity: float = 1.25, seed: int = 1) -> dict[str, float]:
    """每个时刻发布 1 条。RELIABLE：排队、丢了重发；BEST_EFFORT：只发最新的，丢了就丢。返回最后的"数据年龄"（时刻数）。"""
    rng = np.random.default_rng(seed)
    queue: list[int] = []
    rel_last, be_last, credit = -1, -1, 0.0
    be_got = 0
    for t in range(ticks):
        queue.append(t)
        credit += capacity
        while credit >= 1.0:  # 这个时刻能发几次
            credit -= 1.0
            if queue and rng.random() > p:
                rel_last = queue.pop(0)
        if rng.random() > p:  # BEST_EFFORT 只发这一刻的新消息，一次
            be_last = t
            be_got += 1
    return {"reliable_age": ticks - 1 - rel_last, "reliable_queue": len(queue),
            "best_effort_age": ticks - 1 - be_last, "best_effort_recv": be_got / ticks}

**函数 `se2`**：平面上的 3×3 齐次变换：先转 yaw，再平移 (x, y)。

In [ ]:
def se2(x: float, y: float, yaw_deg: float) -> np.ndarray:
    """平面上的 3×3 齐次变换：先转 yaw，再平移 (x, y)。"""
    c, s = np.cos(np.radians(yaw_deg)), np.sin(np.radians(yaw_deg))
    return np.array([[c, -s, x], [s, c, y], [0, 0, 1.0]])

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第三步：RELIABLE 每条平均发几次 = 1 / (1 − p)")
for p in (0.05, 0.35):
    print(f"  丢包率 {p:.2f}：公式 {expected_sends(p):.3f} 次，模拟 {monte_carlo_sends(p):.3f} 次")

In [ ]:
print("\n第三步：队列不爆的条件——发布 1 条/时刻 × 1/(1−p) < 链路 1.25 次/时刻")
for p in (0.05, 0.35):
    need = expected_sends(p)
    print(f"  p = {p:.2f}：需要 {need:.3f} 次/时刻，{'能跟上' if queue_ok(1.0, p, 1.25) else '跟不上，队列越积越长'}")
print(f"  临界丢包率 = 1 − 1/1.25 = {1 - 1 / 1.25:.2f}")

In [ ]:
print("\n动画：跑 200 个时刻，看收到的最新一帧有多旧")
for p in (0.05, 0.35):
    r = simulate_qos(p)
    print(f"  p = {p:.2f}：RELIABLE 最新一帧旧 {r['reliable_age']:.0f} 个时刻（队列里还有 {r['reliable_queue']:.0f} 条）；"
          f"BEST_EFFORT 旧 {r['best_effort_age']:.0f} 个时刻，收到 {r['best_effort_recv']:.0%}")

In [ ]:
print("\n第四步：tf2 坐标变换链 A_T_C = A_T_B · B_T_C（编的位姿）")
base_T_cam = se2(0.2, 0.0, 90.0)  # 相机装在底座前方 0.2 m，转了 90°
p_cam = np.array([1.0, 0.0, 1.0])  # 相机系里正前方 1 m 的点
p_base = base_T_cam @ p_cam
print(f"  相机系 (1, 0) → base_link 系 ({p_base[0]:.2f}, {p_base[1]:.2f})")
back = np.linalg.inv(base_T_cam) @ p_base
print(f"  再用逆变换变回去 → ({back[0]:.2f}, {back[1]:.2f})")

## 3. 代码里才看得到的

- `rng.geometric(1 − p)` 正好是"一直重发直到成功，一共发了几次"的分布，平均值就是 $1/(1-p)$。
- 平面坐标变换用 3×3 齐次矩阵，点要写成 (x, y, 1)；最后那个 1 让平移也能写成矩阵乘法。三维 tf2 里是 4×4，道理一样。

## 4. 自己改一改

- 把丢包率改成 0.20（临界值）和 0.25，跑长一点（`ticks=2000`），看 RELIABLE 的延迟怎么变。
- 再串一个变换（夹爪在相机前方），用两次矩阵乘法把夹爪上的点变到 base_link 系。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 节点像电台，话题像频道；服务一问一答，动作有进度能取消。
> 差网络里控制要新鲜不要完整；ROS 2 不保证实时，实时要自己隔出来。